# AML_01 — Du notebook au laboratoire hybride

**Compétition de travail : [fraud](https://www.kaggle.com/competitions/playground-series-s6e11)**

**Question : Peut-on reproduire la même expérience en local et dans un environnement isolé ?**

7 heures pédagogiques, hors pauses. Les exemples sont des jalons techniques courts ; l’essentiel du temps est consacré aux exercices, revues et expériences. **Version étudiant.**

## Lecture accompagnée

Avant le code, lire avec le professeur le [chapitre 1 du cours rédigé](../lectures/AML_01_cours.md), section par section. Répondre à une question, corriger ensemble, puis retrouver le même raisonnement dans le notebook. Le chapitre de J1 est fourni dans le dossier `lectures`.

## Parcours guidé — Comprendre une prédiction avant de lancer un modèle

**Objectif minimal :** Identifier X, y et une prédiction sur une ligne, puis retrouver le fichier produit. Pas de changement d’hyperparamètre exigé avant cette vérification.

**Idée à retenir :** Une cible est la réponse observée ; une probabilité est une prévision ; un seuil transforme la prévision en décision.

Nous avançons en cinq étapes : **je montre → nous faisons ensemble → vous reproduisez → vous changez une chose → vous expliquez**. Ne pas lancer les autres cellules avant le signal de l’enseignant. Le premier exemple est volontairement minuscule et distinct des données de compétition.

### Exemple au tableau

| Transaction | Vraie classe | Probabilité de fraude |
|---|---:|---:|
| A | 0 | 0,10 |
| B | 1 | 0,70 |
| C | 0 | 0,20 |
| D | 1 | 0,40 |
| E | 0 | 0,60 |
| F | 0 | 0,05 |

Convention : alerte si probabilité ≥ 0,50. Faire la ligne A en entier : 0,10 < 0,50, donc pas d’alerte ; la cible vaut 0, c’est une bonne décision. Faire B ensemble. Laisser chacun compléter C à F avant toute discussion.

### Répondez avant de lancer Python

1. Quelle différence entre la colonne cible et la colonne probabilité ?
2. Quelles transactions déclenchent une alerte ?
3. Combien de fraudes sont ratées ? Combien de fausses alertes ?
4. Pourquoi devons-nous soumettre des probabilités pour la compétition fraude ?

## Préparation et limites

Lire README.md, compétitions et règles. Les données officielles sont téléchargées personnellement et placées dans data/competitions/<slug>/. Aucun remplacement silencieux par des données simulées. Poids préentraînés à préparer avant NLP_02/CV_01/CV_02. Les limites de calcul et d’échanges dépendent du concours et du compte.

## Notions du jour

Question métier : ordonner les transactions par risque de fraude. Le Playground utilise des données synthétiques issues d’un problème réel : ne pas les présenter comme de vraies transactions bancaires. Une ligne = une transaction, cible `is_fraud`. La métrique officielle est la **ROC-AUC**, à maximiser. `submission.csv` contient `id,is_fraud`, avec des probabilités entre 0 et 1.

Une AUC de 0,5 correspond à un classement non discriminant ; elle ne mesure ni le coût financier ni la calibration. L’accuracy peut être trompeuse si les fraudes sont rares. La log-loss et la précision/rappel sont complémentaires, sans remplacer la métrique du concours.

La boucle est : hypothèse locale → test court → entraînement Kaggle → import vérifié → analyse → prochaine hypothèse. Un notebook qui tourne n’est pas une preuve de généralisation.

In [ ]:
from pathlib import Path
import sys,os,json,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import SPECS,read_competition,protocol,run_competition,binary_scores,get_folder
KEY='fraud';SPEC=SPECS[KEY]
QUICK=os.environ.get('MLLAB_QUICK','0')=='1'
LIMIT=int(os.environ.get('MLLAB_LIMIT','1000' if QUICK else '15000'))
IMAGE_LIMIT=int(os.environ.get('MLLAB_IMAGE_LIMIT','24' if QUICK else '240'))
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False})
print('Compétition :',SPEC['slug'],'métrique :',SPEC['metric'],'sens :',SPEC['direction'])
if KEY!='catsdogs':a,test,sample,protocol_record=protocol(KEY)
else:
 from mllab.vision_competition import image_protocol
 image_protocol(limit=IMAGE_LIMIT)
print('Données officielles trouvées. Profil rapide :',QUICK,'; aucun score Kaggle inventé.')

### Retrouver le tableau en Python

Lire les entrées ; prévoir la sortie ; exécuter ; comparer avec le calcul manuel.

In [ ]:
# Vérification du calcul au tableau, distincte des vraies données Kaggle.
from sklearn.metrics import confusion_matrix,roc_auc_score
truth=[0,1,0,1,0,0];prob=np.array([.10,.70,.20,.40,.60,.05])
print('Classes prédites :',(prob>=.5).astype(int))
print('Matrice :',confusion_matrix(truth,prob>=.5))
assert np.isclose(roc_auc_score(truth,prob),.875)
print('ROC-AUC :',roc_auc_score(truth,prob))

## Lire les vrais fichiers et la métrique

**Point d’arrêt avant le calcul 1.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
display(a.head());print('Train officiel',a.shape,'test officiel',test.shape)
print('Fréquence cible',a[SPEC['target']].value_counts(normalize=True).to_dict())
print('Colonnes',a.dtypes.to_dict());print('Valeurs absentes',int(a.isna().sum().sum()))
display(sample.head())
assert list(sample)==['id','is_fraud']
print('ROC-AUC : classer les fraudes au-dessus des non-fraudes ; soumettre des probabilités, pas des classes.')

## Baseline de compétition et premier challenger

**Point d’arrêt avant le calcul 2.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
rows=[]
for model in ['dummy','linear']:
 out,scores,fitted=run_competition(KEY,model=model,limit=LIMIT,hypothesis='Comparer le signal appris à une probabilité constante.')
 rows.append({'model':model,**scores});print('Run',out)
display(pd.DataFrame(rows))
print('Une probabilité constante donne une ROC-AUC de 0,5 ; l’accuracy majoritaire peut paraître haute.')

## Boucle locale → paquet Kaggle → résultats

**Point d’arrêt avant le calcul 3.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
from mllab.competition import export_competition,import_competition_result
from mllab.core import pack_run
exchange=ROOT/'runs/exchange';exchange.mkdir(parents=True,exist_ok=True)
packet=export_competition(KEY,exchange/'experiment.zip',model='linear',limit=LIMIT)
archive=pack_run(out,exchange/'result_local.zip');received=import_competition_result(archive)
print('Paquet',packet,'résultat importé',received)
print('Ce round-trip local vérifie le contrat. Exécuter ensuite templates/KAGGLE_RUNNER.ipynb sur Kaggle.')

## Extensions facultatives — après validation du socle

Ces pistes servent aux étudiants déjà à l’aise. Elles ne sont pas à réaliser toutes ni nécessaires pour accéder à la séance suivante. Le travail obligatoire est celui indiqué dans le parcours guidé et la fiche de séance.

1. Écrire le QUID de la compétition et expliquer pourquoi on prédit une probabilité plutôt qu’une décision 0/1.
2. Changer un seul paramètre de régularisation, mesurer score et durée sur le même sous-échantillon.
3. Faire tourner le paquet sur Kaggle, récupérer result.zip et vérifier son import ; conserver une preuve de l’exécution distante.

Pour chaque essai : hypothèse avant calcul, code/configuration, empreinte du split, métrique officielle, durée, budget, conclusion, prochaine expérience. Chaque membre doit pouvoir expliquer son apport.

**Point d’arrêt avant le calcul 4.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
HYPOTHESE = 'À écrire avant le prochain calcul'
CONCLUSION = 'À compléter avec une preuve et une limite'
# Ajouter ici une expérience limitée, puis expliquer la décision de la conserver ou non.

## Questions de sortie

- Une accuracy de 90 % impose-t-elle une log-loss unique ?
- Quelle information permet de retrouver exactement les données utilisées ?

**Remise :** un run traçable, une conclusion, un artefact ou une soumission valide et une prochaine hypothèse. Aucun score public ni prix garanti.